# 6C. Per-Exercise Keypoint Weighting Sweep

This notebook follows the frozen `6B` sequence-length sweep and tests whether exercise-specific keypoint emphasis can reduce error without rebuilding the Stage 5 pose-sequence dataset.


## Reason, Approach, Result Interpretation

**Reason**
- `6B` showed that sequence length matters, but it did not reduce error enough on its own.
- The current TCN still treats all keypoints equally, even though different exercises depend on different body regions.

**Approach**
- Reuse the best `seq_len` found in `6B` for each target exercise.
- Keep the same TCN family and augmentation recipe.
- Apply exercise-specific keypoint weighting inside the trainer so the experiment remains compatible with the existing Stage 5 sequence files.

**Result interpretation**
- Lower `MAE` and higher `Within-1` than the `6B` baseline indicate that keypoint emphasis is helping.
- If the weighted runs remain close to the unweighted baseline, the next bottleneck is likely the model formulation rather than the pose representation alone.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Environment Setup

**Why this section exists**
- The weighting sweep needs the updated Stage 6 trainer, the baseline-comparison script, and the current full `pose_sequence_index.csv` in the same Colab session.

**Approach**
- Mount Drive.
- Sync the current repo copies of the trainer and comparison script into the Drive project.
- Resolve the sequence index that was rebuilt from the full pose-feature index.

**How to interpret the result**
- If the script paths print correctly and `SEQUENCE_INDEX exists = True`, the environment is ready.
- If the index is missing, rerun Stage 5 before starting this notebook.


In [ ]:
from pathlib import Path
import shutil

CODE_ROOT = Path('/content/CV_Image_pose_detection')
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/FinalProjectCV/CV_Image_pose_detection')

TRAINER_REL = Path('artifacts/3_Modeling/train_pose_count_tcn.py')
COMPARE_REL = Path('artifacts/3_Modeling/compare_count_run_to_baseline.py')
TRAINER_SRC = CODE_ROOT / TRAINER_REL
TRAINER_DST = DRIVE_PROJECT_ROOT / TRAINER_REL
COMPARE_SRC = CODE_ROOT / COMPARE_REL
COMPARE_DST = DRIVE_PROJECT_ROOT / COMPARE_REL

if TRAINER_SRC.exists():
    TRAINER_DST.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(TRAINER_SRC, TRAINER_DST)

if COMPARE_SRC.exists():
    COMPARE_DST.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(COMPARE_SRC, COMPARE_DST)

ANNOTATION_DIR = DRIVE_PROJECT_ROOT / 'Data/LLSP/annotation_cleaned'
SEQUENCE_INDEX = ANNOTATION_DIR / 'pose_sequence_index.csv'

print('TRAINER_DST =', TRAINER_DST)
print('COMPARE_DST =', COMPARE_DST)
print('SEQUENCE_INDEX =', SEQUENCE_INDEX)
print('SEQUENCE_INDEX exists =', SEQUENCE_INDEX.exists())


## Dataset Coverage Check

**Why this section exists**
- The weighting sweep depends on the full Stage 5 rebuild, including squat.
- Each target exercise needs nonzero `train` and `valid` rows.

**Approach**
- Read `pose_sequence_index.csv`.
- Display the split counts for the exercises selected from `6B`.

**How to interpret the result**
- If a target exercise is missing, the issue is still upstream in Stage 5 rather than in the weighted TCN experiment.


In [ ]:
import pandas as pd

TARGET_EXERCISES = ['bench_pressing', 'pommelhorse', 'pull_up', 'push_up', 'squat']

seq_df = pd.read_csv(SEQUENCE_INDEX)
counts_df = seq_df.groupby(['type', 'split']).size().unstack(fill_value=0).sort_index()
display(counts_df.loc[TARGET_EXERCISES])


## Weighting Design

**Why this section exists**
- `6B` already selected the best sequence length for each promising exercise.
- This stage changes only one additional factor: exercise-specific keypoint emphasis.

**Approach**
- Reuse the strongest `seq_len` found in `6B`.
- Train one new weighted run per exercise using `--keypoint-profile auto_exercise_v1`.
- Compare each weighted run back to its unweighted `6B` reference.

**How to interpret the result**
- If the weighted run beats the corresponding `6B` run on `MAE` or `Within-1`, the pose representation is benefiting from more selective body-region emphasis.
- If the weighted run does not improve, the next lever is likely a different model formulation rather than more hand-crafted pose weighting.


In [ ]:
import pandas as pd

EXERCISE_CONFIGS = [
    {
        'exercise': 'bench_pressing',
        'seq_len': 192,
        'baseline_run': 'pose_count_tcn_bench_pressing_seq192',
        'weighted_run': 'pose_count_tcn_bench_pressing_seq192_weightedv1',
    },
    {
        'exercise': 'pommelhorse',
        'seq_len': 192,
        'baseline_run': 'pose_count_tcn_pommelhorse_seq192',
        'weighted_run': 'pose_count_tcn_pommelhorse_seq192_weightedv1',
    },
    {
        'exercise': 'pull_up',
        'seq_len': 192,
        'baseline_run': 'pose_count_tcn_pull_up_seq192',
        'weighted_run': 'pose_count_tcn_pull_up_seq192_weightedv1',
    },
    {
        'exercise': 'push_up',
        'seq_len': 128,
        'baseline_run': 'pose_count_tcn_push_up_seq128',
        'weighted_run': 'pose_count_tcn_push_up_seq128_weightedv1',
    },
    {
        'exercise': 'squat',
        'seq_len': 256,
        'baseline_run': 'pose_count_tcn_squat_seq256',
        'weighted_run': 'pose_count_tcn_squat_seq256_weightedv1',
    },
]

WEIGHT_PROFILE = 'auto_exercise_v1'
PROFILE_STRENGTH = 1.0

EPOCHS = 80
BATCH_SIZE = 16
LR = 1e-3
WEIGHT_DECAY = 1e-4
CHANNELS = 96
KERNEL_SIZE = 3
NUM_BLOCKS = 4
DROPOUT = 0.2
PATIENCE = 15
LOSS = 'l1'
EVAL_TRANSFORM = 'raw'
SELECTION_METRIC = 'mae'
SAMPLER = 'balanced_count'
TIME_WARP_RANGE = 0.12
FEATURE_NOISE_STD = 0.02
FRAME_DROPOUT_PROB = 0.03

display(pd.DataFrame(EXERCISE_CONFIGS))
print('WEIGHT_PROFILE =', WEIGHT_PROFILE)
print('PROFILE_STRENGTH =', PROFILE_STRENGTH)


## Training Execution

**Why this section exists**
- This is the actual weighted run stage.
- It launches one exercise-specific TCN per selected exercise, using the best `seq_len` from `6B`.

**Approach**
- Keep the architecture and augmentation fixed.
- Add only `--keypoint-profile` and `--profile-strength`.
- Keep failures non-fatal so one bad run does not block the full comparison.

**How to interpret the result**
- A clean sweep means every weighted run finished and wrote a new artifact folder.
- If one exercise fails, debug that run separately without discarding the rest of the sweep.


In [ ]:
import subprocess
import pandas as pd

training_failures = []
for cfg in EXERCISE_CONFIGS:
    cmd = [
        'python', str(TRAINER_DST),
        '--project-dir', str(DRIVE_PROJECT_ROOT),
        '--index-csv', str(SEQUENCE_INDEX),
        '--run-name', cfg['weighted_run'],
        '--exercise', cfg['exercise'],
        '--seq-len', str(cfg['seq_len']),
        '--epochs', str(EPOCHS),
        '--batch-size', str(BATCH_SIZE),
        '--lr', str(LR),
        '--weight-decay', str(WEIGHT_DECAY),
        '--channels', str(CHANNELS),
        '--kernel-size', str(KERNEL_SIZE),
        '--num-blocks', str(NUM_BLOCKS),
        '--dropout', str(DROPOUT),
        '--patience', str(PATIENCE),
        '--loss', LOSS,
        '--eval-transform', EVAL_TRANSFORM,
        '--selection-metric', SELECTION_METRIC,
        '--sampler', SAMPLER,
        '--time-warp-range', str(TIME_WARP_RANGE),
        '--feature-noise-std', str(FEATURE_NOISE_STD),
        '--frame-dropout-prob', str(FRAME_DROPOUT_PROB),
        '--keypoint-profile', WEIGHT_PROFILE,
        '--profile-strength', str(PROFILE_STRENGTH),
        '--device', 'cuda',
    ]
    print('\nRunning:', ' '.join(cmd))
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as exc:
        training_failures.append({
            'exercise': cfg['exercise'],
            'seq_len': cfg['seq_len'],
            'weighted_run': cfg['weighted_run'],
            'returncode': exc.returncode,
        })
        print(f"FAILED: {cfg['exercise']} (returncode={exc.returncode})")

if training_failures:
    display(pd.DataFrame(training_failures))
else:
    print('All weighted runs completed.')


## Raw Metric Review

**Why this section exists**
- The first check should compare the direct validation metrics of the weighted runs against the corresponding best unweighted `6B` runs.

**Approach**
- Load `metrics_summary.json` for both variants.
- Compare `valid_mae`, `valid_rmse`, and `valid_within_1` within each exercise.

**How to interpret the result**
- Lower `valid_mae` and higher `valid_within_1` in the weighted row mean the keypoint profile helped.
- If the weighted row is flat or worse, the next bottleneck is likely not simple keypoint emphasis.


In [ ]:
import json
import pandas as pd

rows = []
for cfg in EXERCISE_CONFIGS:
    for variant, run_name in [('baseline_6B', cfg['baseline_run']), ('weighted_v1', cfg['weighted_run'])]:
        metrics_path = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs' / run_name / 'metrics_summary.json'
        if not metrics_path.exists():
            continue
        with open(metrics_path, 'r', encoding='utf-8') as f:
            metrics = json.load(f)
        rows.append({
            'exercise': cfg['exercise'],
            'seq_len': cfg['seq_len'],
            'variant': variant,
            'run_name': run_name,
            'best_epoch': metrics.get('best_epoch'),
            'valid_mae': metrics['valid_metrics']['mae'],
            'valid_rmse': metrics['valid_metrics']['rmse'],
            'valid_within_1': metrics['valid_metrics']['within_1'],
        })

raw_df = pd.DataFrame(rows)
if raw_df.empty:
    print('No metrics_summary.json files found yet.')
else:
    display(raw_df.sort_values(['exercise', 'variant']))
    pivot_df = raw_df.pivot(index=['exercise', 'seq_len'], columns='variant', values=['valid_mae', 'valid_within_1'])
    pivot_df.columns = ['_'.join(col).strip() for col in pivot_df.columns.values]
    pivot_df = pivot_df.reset_index()
    if 'valid_mae_baseline_6B' in pivot_df.columns and 'valid_mae_weighted_v1' in pivot_df.columns:
        pivot_df['delta_mae_weighted_minus_baseline6B'] = pivot_df['valid_mae_weighted_v1'] - pivot_df['valid_mae_baseline_6B']
    if 'valid_within_1_baseline_6B' in pivot_df.columns and 'valid_within_1_weighted_v1' in pivot_df.columns:
        pivot_df['delta_within_1_weighted_minus_baseline6B'] = pivot_df['valid_within_1_weighted_v1'] - pivot_df['valid_within_1_baseline_6B']
    display(pivot_df.sort_values('exercise'))


## Baseline-Comparison Review

**Why this section exists**
- Lower raw `MAE` is useful, but we still need to check whether the weighted runs improve value over the trivial train-split baseline and over the best unweighted `6B` reference.

**Approach**
- Ensure each weighted run has a `baseline_comparison_summary.json` file.
- Load the summary for the unweighted `6B` run and the new weighted run.
- Compare both their trivial-baseline deltas and their direct weighted-vs-unweighted gap.

**How to interpret the result**
- More negative `delta_mae_vs_trivial` is better.
- More positive `delta_within_1_vs_trivial` is better.
- Negative `delta_mae_weighted_minus_baseline6B` and positive `delta_within_1_weighted_minus_baseline6B` indicate that the weighting profile improved the exercise beyond the best `6B` temporal setting.


In [ ]:
import subprocess
import json
import pandas as pd

comparison_failures = []
for cfg in EXERCISE_CONFIGS:
    run_dir = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs' / cfg['weighted_run']
    pred_path = run_dir / 'predictions.csv'
    if not pred_path.exists():
        continue
    summary_path = run_dir / 'baseline_comparison_summary.json'
    if summary_path.exists():
        continue
    cmd = [
        'python', str(COMPARE_DST),
        '--index-csv', str(SEQUENCE_INDEX),
        '--predictions-csv', str(pred_path),
        '--exercise', cfg['exercise'],
    ]
    print('\nComparing:', ' '.join(cmd))
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as exc:
        comparison_failures.append({
            'exercise': cfg['exercise'],
            'seq_len': cfg['seq_len'],
            'weighted_run': cfg['weighted_run'],
            'returncode': exc.returncode,
        })

rows = []
for cfg in EXERCISE_CONFIGS:
    for variant, run_name in [('baseline_6B', cfg['baseline_run']), ('weighted_v1', cfg['weighted_run'])]:
        summary_path = DRIVE_PROJECT_ROOT / 'artifacts/3_Modeling/training_outputs' / run_name / 'baseline_comparison_summary.json'
        if not summary_path.exists():
            continue
        with open(summary_path, 'r', encoding='utf-8') as f:
            summary = json.load(f)
        rows.append({
            'exercise': cfg['exercise'],
            'seq_len': cfg['seq_len'],
            'variant': variant,
            'run_name': run_name,
            'model_mae': summary['model_metrics']['mae'],
            'baseline_mae': summary['baseline_metrics']['mae'],
            'delta_mae_vs_trivial': summary['delta_vs_baseline']['mae'],
            'model_within_1': summary['model_metrics']['within_1'],
            'baseline_within_1': summary['baseline_metrics']['within_1'],
            'delta_within_1_vs_trivial': summary['delta_vs_baseline']['within_1'],
            'model_beats_baseline_rows': summary['row_level']['model_beats_baseline'],
            'valid_rows': summary['row_level']['valid_rows'],
        })

compare_df = pd.DataFrame(rows)
if not compare_df.empty:
    display(compare_df.sort_values(['exercise', 'variant']))
    compare_pivot = compare_df.pivot(index=['exercise', 'seq_len'], columns='variant', values=['model_mae', 'model_within_1', 'delta_mae_vs_trivial', 'delta_within_1_vs_trivial'])
    compare_pivot.columns = ['_'.join(col).strip() for col in compare_pivot.columns.values]
    compare_pivot = compare_pivot.reset_index()
    if 'model_mae_baseline_6B' in compare_pivot.columns and 'model_mae_weighted_v1' in compare_pivot.columns:
        compare_pivot['delta_mae_weighted_minus_baseline6B'] = compare_pivot['model_mae_weighted_v1'] - compare_pivot['model_mae_baseline_6B']
    if 'model_within_1_baseline_6B' in compare_pivot.columns and 'model_within_1_weighted_v1' in compare_pivot.columns:
        compare_pivot['delta_within_1_weighted_minus_baseline6B'] = compare_pivot['model_within_1_weighted_v1'] - compare_pivot['model_within_1_baseline_6B']
    display(compare_pivot.sort_values('exercise'))
else:
    print('No baseline comparison summaries found yet.')

if comparison_failures:
    display(pd.DataFrame(comparison_failures))
